In [1]:

# ============================================================
#  THE GIFT SHOP — AI-Powered Gift Recommendation Engine
#  Created by Gaurab & Vidushi
#  Final AI Project — Prof. Armen Pischsdotchian
# ============================================================

!pip install ibm-watsonx-ai gradio markdown openai -q
print("✅ All packages ready! Starting the app...")

import os
import markdown as md_lib
import gradio as gr
from openai import OpenAI
import re

# ============================================================
#  API SETUP — Groq (Llama model)
# ============================================================
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
os.environ["MODEL"] = "llama-3.1-8b-instant"

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

# ============================================================
#  OCCASION ASSETS (Images Fixed with Reliable Links)
# ============================================================
OCCASION_ASSETS = {
    # --- Annual Holidays ---
    "💕 Valentine's Day": {"primary": "#f43f5e", "hero_img": "https://images.unsplash.com/photo-1516589178581-6cd7833ae3b2?w=800&q=80", "amazon_tag": "valentines+day+gifts"},
    "🐰 Easter": {"primary": "#f472b6", "hero_img": "https://images.unsplash.com/photo-1522337360788-8b13dee7a37e?w=800&q=80", "amazon_tag": "easter+basket+gifts"},
    "🌸 Mother's Day": {"primary": "#ec4899", "hero_img": "https://images.unsplash.com/photo-1606787364406-a3cdf06c6d0c?w=800&q=80", "amazon_tag": "mothers+day+gifts"},
    "👔 Father's Day": {"primary": "#3b82f6", "hero_img": "https://images.unsplash.com/photo-1472099645785-5658abf4ff4e?w=800&q=80", "amazon_tag": "fathers+day+gifts"},
    "👵 Grandparents' Day": {"primary": "#8b5cf6", "hero_img": "https://images.unsplash.com/photo-1511895426328-dc8714191300?w=800&q=80", "amazon_tag": "grandparents+day+gifts"},
    "🎃 Halloween": {"primary": "#ea580c", "hero_img": "https://images.unsplash.com/photo-1508361001413-7a9dca21d08a?w=800&q=80", "amazon_tag": "halloween+gifts+treats"},
    "🦃 Thanksgiving": {"primary": "#d97706", "hero_img": "https://images.unsplash.com/photo-1511795409834-ef04bbd61622?w=800&q=80", "amazon_tag": "thanksgiving+host+gifts"},
    "🕎 Hanukkah": {"primary": "#818cf8", "hero_img": "https://images.unsplash.com/photo-1576020799627-aeac74d58064?w=800&q=80", "amazon_tag": "hanukkah+gifts"},
    "🎄 Christmas": {"primary": "#22c55e", "hero_img": "https://images.unsplash.com/photo-1512389142860-9c449e58a543?w=800&q=80", "amazon_tag": "christmas+gifts"},
    "🕯️ Kwanzaa": {"primary": "#dc2626", "hero_img": "https://images.unsplash.com/photo-1547919307-1ecb10702e6f?w=800&q=80", "amazon_tag": "kwanzaa+gifts"},

    # --- Life Milestones ---
    "🎂 Birthdays": {"primary": "#f59e0b", "hero_img": "https://images.unsplash.com/photo-1464349095431-e9a21285b5f3?w=800&q=80", "amazon_tag": "birthday+gift+ideas"},
    "💍 Weddings": {"primary": "#fbcfe8", "hero_img": "https://images.unsplash.com/photo-1519225421980-715cb0215aed?w=800&q=80", "amazon_tag": "wedding+gifts"},
    "🍼 Baby Showers": {"primary": "#60a5fa", "hero_img": "https://images.unsplash.com/photo-1519689680058-324335c77eba?w=800&q=80", "amazon_tag": "baby+shower+gifts"},
    "🎓 Graduations": {"primary": "#1e3a8a", "hero_img": "https://images.unsplash.com/photo-1627556704302-624286467c65?w=800&q=80", "amazon_tag": "graduation+gifts"},
    "🥂 Anniversaries": {"primary": "#be123c", "hero_img": "https://images.unsplash.com/photo-1543157145-f78c636d023d?w=800&q=80", "amazon_tag": "anniversary+gifts"},
    "🏡 Housewarmings": {"primary": "#059669", "hero_img": "https://images.unsplash.com/photo-1556912167-f556f1f39fdf?w=800&q=80", "amazon_tag": "housewarming+gifts"},
    "🌴 Retirement": {"primary": "#0891b2", "hero_img": "https://images.unsplash.com/photo-1499793983690-e29da59ef1c2?w=800&q=80", "amazon_tag": "retirement+gifts"},

    # --- Social & Professional ---
    "🍎 Teacher Appreciation": {"primary": "#eab308", "hero_img": "https://images.unsplash.com/photo-1503676260728-1c00da094a0b?w=800&q=80", "amazon_tag": "teacher+appreciation+gifts"},
    "💼 Admin Professionals Day": {"primary": "#4f46e5", "hero_img": "https://images.unsplash.com/photo-1497032628192-86f99bcd76bc?w=800&q=80", "amazon_tag": "office+gifts+for+coworkers"},
    "🍷 Hostess Gifts": {"primary": "#14b8a6", "hero_img": "https://images.unsplash.com/photo-1512163143273-bff31aeb7c09?w=800&q=80", "amazon_tag": "hostess+gifts"},
    "🍲 Get Well Soon": {"primary": "#10b981", "hero_img": "https://images.unsplash.com/photo-1506126613408-eca07ce68773?w=800&q=80", "amazon_tag": "get+well+soon+care+package"},
    "🕊️ Sympathy/Funerals": {"primary": "#64748b", "hero_img": "https://images.unsplash.com/photo-1494537176433-7a3c4ef2046f?w=800&q=80", "amazon_tag": "sympathy+gifts"},
    "🐘 White Elephant / Secret Santa": {"primary": "#b91c1c", "hero_img": "https://images.unsplash.com/photo-1513201099705-a9746e1e201f?w=800&q=80", "amazon_tag": "white+elephant+gifts+funny"},

    # --- International & Cultural ---
    "🧧 Lunar New Year": {"primary": "#e11d48", "hero_img": "https://images.unsplash.com/photo-1551041777-ed277b8dd348?w=800&q=80", "amazon_tag": "lunar+new+year+gifts"},
    "🌙 Eid al-Fitr / Eid al-Adha": {"primary": "#047857", "hero_img": "https://images.unsplash.com/photo-1585036156171-384164a8c675?w=800&q=80", "amazon_tag": "eid+gifts"},
    "🪔 Diwali": {"primary": "#fbbf24", "hero_img": "https://images.unsplash.com/photo-1514222134-b57cbb8ce073?w=800&q=80", "amazon_tag": "diwali+gift+hampers"},

    # --- Default ---
    "🎁 Just Because": {"primary": "#1abc9c", "hero_img": "https://images.unsplash.com/photo-1607344645866-009c320b63e0?w=800&q=80", "amazon_tag": "surprise+gifts+thoughtful"}
}

OCCASION_NAMES = list(OCCASION_ASSETS.keys())

# ============================================================
#  GIFT CART STATE
# ============================================================
gift_cart = []
cart_id_counter = [1]

def get_cart_id():
    cid = cart_id_counter[0]
    cart_id_counter[0] += 1
    return cid

# ============================================================
#  AI GIFT LOGIC ENGINE (STRICT BUDGET ENFORCEMENT)
# ============================================================
def get_gift_strategy(interview_answers, messy_facts):
    system_message = (
        "You are a world-class Gift-Logic Reasoning Agent. You MUST follow instructions perfectly. "
        "Always output ONLY a markdown table. No preamble, no extra text."
    )
    user_prompt = f"""
    ANALYSIS MISSION:
    1. FACTS: '{messy_facts}'
    2. STRICT BUDGET CONSTRAINT: The budget is EXACTLY {interview_answers['budget']}. You MUST provide items where the estimated price falls explicitly inside this range. If the budget says '$500+', EVERY item must cost more than $500. If it says 'Under $25', EVERY item must cost less than $25. Do NOT violate this rule.
    3. OCCASION: {interview_answers.get('occasion', 'a special occasion')}
    4. REASON: Persona ({interview_answers['persona']}), Goal ({interview_answers['goal']}). Find 3-5 unique gifts.

    FORMAT: Output ONLY a Markdown table with these exact columns:
    | Gift | Logic Rationale | Price Est | Amazon Search |
    """
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_prompt}
            ],
            model=os.environ["MODEL"],
            temperature=0.3
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# ============================================================
#  HTML HELPERS & UI ORCHESTRATOR
# ============================================================
def update_hero_image(occasion):
    assets = OCCASION_ASSETS.get(occasion, OCCASION_ASSETS["🎁 Just Because"])
    return f"""
    <div style="width:100%; overflow:hidden; border-radius:8px; box-shadow:0 10px 30px rgba(0,0,0,0.1);">
      <img src="{assets['hero_img']}" alt="{occasion}"
           style="width:100%; height:380px; object-fit:cover; filter:brightness(0.9); transition:all 0.5s ease;"
           onmouseover="this.style.filter='brightness(1.0)'"
           onmouseout="this.style.filter='brightness(0.9)'">
      <div style="background:{assets['primary']}; padding:12px 16px; text-align:center;">
        <span style="color:white !important; font-size:13px; font-weight:700; letter-spacing:2px; text-transform:uppercase;">
          ✦ {occasion} ✦
        </span>
      </div>
    </div>
    """

def ui_logic_orchestrator(occasion, budget, persona, goal, context):
    if not occasion:
        return "<div style='color:#ef4444; font-weight:bold;'>Please select an occasion first.</div>"

    assets = OCCASION_ASSETS.get(occasion, OCCASION_ASSETS["🎁 Just Because"])
    interview_data = {"budget": budget, "persona": persona, "goal": goal, "occasion": occasion}
    enhanced_context = f"Occasion: {occasion}. Persona: {persona}. Goal: {goal}. Details: {context}"

    amazon_url = f"https://www.amazon.com/s?k={assets['amazon_tag'].replace(' ', '+')}"

    header_html = f"""
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:24px;">
      <h3 style="color:#0f172a !important; font-weight:800; letter-spacing:1px; text-transform:uppercase; margin:0; font-size:16px;">✧ AI Recommendations</h3>
      <a href="{amazon_url}" target="_blank" style="background:#0284c7 !important; color:white !important; padding:10px 20px; text-decoration:none; font-weight:bold; font-size:13px; border-radius:4px;">
        🛍️ Shop Category on Amazon →
      </a>
    </div>
    """

    try:
        raw_markdown = get_gift_strategy(interview_data, enhanced_context)
        lines = [l.strip() for l in raw_markdown.strip().split('\n') if l.strip()]
        table_lines = [l for l in lines if l.startswith('|')]

        if table_lines:
            processed_rows = []
            for row in table_lines:
                if re.match(r'\|.*-{3,}.*\|', row):
                    processed_rows.append(row)
                    continue

                cells = [c.strip() for c in row.split('|')]
                if len(cells) > 4 and cells[1].strip().lower() != 'gift':
                    raw_term = cells[4].replace('"', '').replace("'", "").strip()
                    search_term = re.sub(r'<[^>]+>', '', raw_term)

                    if search_term:
                        item_url = f"https://www.amazon.com/s?k={search_term.replace(' ', '+')}"
                        cells[4] = f"[Search Amazon →]({item_url})"

                    row = "| " + " | ".join(cells[1:-1]) + " |"
                processed_rows.append(row)

            table_md = "\n".join(processed_rows)
            html_table = md_lib.markdown(table_md, extensions=['tables'])

            return f"""
            <div style="background:#ffffff; border-left:6px solid {assets['primary']}; padding:32px; box-shadow:0 4px 15px rgba(0,0,0,0.05); margin-top:10px; border-radius:4px;">
              {header_html}
              <style>
                .gift-table table {{ width:100%; border-collapse:collapse; font-size:14px; color: #0f172a !important; }}
                .gift-table th {{ background:#f8fafc !important; padding:13px 15px; text-align:left; text-transform:uppercase; font-size:12px !important; color:#0f172a !important; font-weight:800 !important; border-bottom:2px solid #cbd5e1 !important; }}
                .gift-table td {{ padding:13px 15px; border-bottom:1px solid #e2e8f0; color:#334155 !important; font-weight:500; line-height:1.6; }}
                .gift-table a {{ color:#0284c7 !important; font-weight:bold; text-decoration:underline; }}
              </style>
              <div class="gift-table">{html_table}</div>
              <p style="margin-top:16px; font-size:13px; color:#0f172a !important; text-align:right; font-weight:bold;">✦ Copy a gift idea from above and add it to your 🛒 Gift Cart below!</p>
            </div>"""
        else:
            return f"""
            <div style="background:#ffffff; border-left:6px solid {assets['primary']}; padding:32px; box-shadow:0 4px 15px rgba(0,0,0,0.05); margin-top:10px; border-radius:4px;">
              {header_html}
              <pre style='color:#0f172a !important; font-weight:500; white-space: pre-wrap; font-family: inherit;'>{raw_markdown}</pre>
            </div>"""
    except Exception as e:
        return f"<div style='color:#dc2626; padding:20px; font-weight:bold;'>⚠️ Error generating ideas: {str(e)}</div>"

# ============================================================
#  CART FUNCTIONS
# ============================================================
def add_to_cart(occasion, for_whom, gift_idea, note):
    if not gift_idea.strip():
        return render_cart(), "<span style='color:#ef4444; font-weight:bold;'>⚠️ Please paste a gift idea to save.</span>"
    item = {
        "id": get_cart_id(),
        "occasion": occasion or "🎁 Just Because",
        "for": for_whom or "Someone Special",
        "idea": gift_idea,
        "note": note or ""
    }
    gift_cart.append(item)
    return render_cart(), f"<span style='color:#10b981; font-weight:bold;'>✅ Saved '{gift_idea}' to cart!</span>"

def remove_from_cart(item_id_str):
    try:
        item_id = int(item_id_str)
        gift_cart[:] = [i for i in gift_cart if i["id"] != item_id]
        return render_cart(), f"<span style='color:#0f172a; font-weight:bold;'>🗑 Removed item ID {item_id}.</span>"
    except:
        return render_cart(), "<span style='color:#ef4444; font-weight:bold;'>⚠️ Invalid ID.</span>"

def clear_all_cart():
    gift_cart.clear()
    return render_cart(), "<span style='color:#0f172a; font-weight:bold;'>🛒 Cart cleared.</span>"

def render_cart():
    if not gift_cart:
        return """<div style='background:#f8fafc; border:2px dashed #cbd5e1; padding:40px; text-align:center; border-radius:8px;'>
          <div style='font-size:32px; margin-bottom:8px;'>🛒</div>
          <p style='color:#475569 !important; font-weight:600; font-size:15px;'>Your gift cart is empty.<br>Copy an AI idea from above and save it here!</p>
        </div>"""

    rows = ""
    for item in gift_cart:
        color = OCCASION_ASSETS.get(item['occasion'], OCCASION_ASSETS["🎁 Just Because"])["primary"]
        amazon_link = f"https://www.amazon.com/s?k={item['idea'].replace(' ', '+')}"
        rows += f"""
        <tr style='border-bottom:1px solid #e2e8f0;'>
          <td style='padding:14px 16px; font-weight:800; color:#0f172a !important;'>{item['id']}</td>
          <td style='padding:14px 16px;'><span style='background:{color}22; color:{color} !important; padding:4px 12px; border-radius:20px; font-size:12px; font-weight:800;'>{item['occasion']}</span></td>
          <td style='padding:14px 16px; color:#334155 !important; font-weight:600;'>{item['for']}</td>
          <td style='padding:14px 16px; color:#0f172a !important; font-weight:800;'>{item['idea']}</td>
          <td style='padding:14px 16px; color:#475569 !important; font-style:italic;'>{item['note']}</td>
          <td style='padding:14px 16px;'><a href="{amazon_link}" target="_blank" style='background:#0f172a; color:white !important; padding:8px 14px; text-decoration:none; font-size:12px; font-weight:bold; border-radius:4px;'>Shop →</a></td>
        </tr>"""

    return f"""
    <div style='background:#ffffff; border:1px solid #cbd5e1; border-radius:8px; overflow:hidden; box-shadow:0 4px 15px rgba(0,0,0,0.05);'>
      <div style='background:#0f172a; padding:16px 20px;'><span style='color:#ffffff !important; font-weight:bold; text-transform:uppercase; letter-spacing:1px;'>🛒 Gift Cart ({len(gift_cart)} items)</span></div>
      <table style='width:100%; border-collapse:collapse; text-align:left; font-size:14px;'>
        <tr style='background:#f8fafc; text-transform:uppercase; font-size:11px; color:#0f172a !important; border-bottom:2px solid #cbd5e1;'>
          <th style='padding:14px 16px; font-weight:800;'>ID</th><th style='padding:14px 16px; font-weight:800;'>Occasion</th><th style='padding:14px 16px; font-weight:800;'>For</th><th style='padding:14px 16px; font-weight:800;'>Idea</th><th style='padding:14px 16px; font-weight:800;'>Notes</th><th style='padding:14px 16px; font-weight:800;'>Action</th>
        </tr>
        {rows}
      </table>
    </div>"""

# ============================================================
#  CSS & UI BUILD
# ============================================================
luxury_css = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700;900&family=DM+Sans:wght@400;500;600;700;800&display=swap');

/* Force Light Theme Globally */
body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background-color: #fcfbf9 !important;
    color: #0f172a !important;
}

h1, h2, h3, h4, p, span, div, label, td, th { color: #0f172a !important; }
h1, h2 { font-family: 'Playfair Display', serif !important; font-weight: 900 !important; }

/* FIX ALL INPUTS, DROPDOWNS, AND TEXTAREAS */
input, textarea, select, .gr-text-input, .gr-input, .gr-dropdown, .gr-box {
    background-color: #ffffff !important;
    color: #0f172a !important;
    font-weight: 500 !important;
    border: 1px solid #cbd5e1 !important;
}

::placeholder { color: #94a3b8 !important; opacity: 1 !important; }

ul.options, li.item {
    background-color: #ffffff !important;
    color: #0f172a !important;
}

/* FIX BLOG ACCORDIONS */
.gr-accordion {
    background-color: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: 8px !important;
    overflow: hidden !important;
}
.gr-accordion-header { background-color: #f8fafc !important; color: #0f172a !important; font-weight: 800 !important; padding: 15px !important; }
.gr-accordion-content { background-color: #ffffff !important; color: #334155 !important; padding: 20px !important; }
.gr-accordion-content p { color: #334155 !important; line-height: 1.8 !important; font-size: 15px !important; }
.gr-accordion-content strong { color: #0f172a !important; font-weight: 800 !important; }

/* FIX BUTTONS */
button.primary { background-color: #0f172a !important; color: white !important; border: none !important; font-weight:bold !important; }
button.primary:hover { background-color: #1e293b !important; }

/* Animations for the Hero Section */
@keyframes float { 0% { transform: translateY(0px); } 50% { transform: translateY(-15px); } 100% { transform: translateY(0px); } }
@keyframes float-delay { 0% { transform: translateY(0px); } 50% { transform: translateY(-10px); } 100% { transform: translateY(0px); } }
"""

js_func = "function(){ document.body.classList.remove('dark'); }"

with gr.Blocks(css=luxury_css, js=js_func) as demo:

    # ── NAV BAR ──
    gr.HTML("""
    <div id="home" style="display:flex; justify-content:space-between; align-items:center; padding:20px 0 24px; border-bottom:2px solid #0f172a; margin-bottom:40px;">
      <div style="font-family:'Playfair Display',serif; font-size:26px; font-weight:900; color:#0f172a !important;">🎁 The Gift Shop</div>
      <div style="display:flex; gap:32px; font-size:13px; font-weight:700; text-transform:uppercase;">
        <a onclick="document.getElementById('home').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; text-decoration:none; color:#0f172a !important;">Home</a>
        <a onclick="document.getElementById('generator').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; text-decoration:none; color:#0f172a !important;">Generator</a>
        <a onclick="document.getElementById('cart').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; text-decoration:none; color:#0f172a !important;">Cart</a>
        <a onclick="document.getElementById('blog').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; text-decoration:none; color:#0f172a !important;">Blog</a>
        <a onclick="document.getElementById('contact').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; text-decoration:none; color:#0f172a !important;">Contact Us</a>
      </div>
      <div style="font-size:13px; color:#0f172a !important; font-weight:800;">By Gaurab & Vidushi | Prof. Armen Pischsdotchian</div>
    </div>
    """)

    # ── HOMEPAGE HERO WITH ANIMATED IMAGES & LLAMA MENTION ──
    gr.HTML("""
    <div style="display: flex; flex-wrap: wrap; align-items: center; gap: 40px; text-align: left; padding: 60px 40px; background-color: #ffffff; margin-bottom: 60px; border-radius: 8px; border: 1px solid #e2e8f0; box-shadow: 0 4px 20px rgba(0,0,0,0.03);">
        <div style="flex: 1 1 500px;">
            <div style="font-size:12px; letter-spacing:3px; color:#10b981 !important; text-transform:uppercase; margin-bottom:16px; font-weight:800;">Welcome to</div>
            <h1 style="font-size:3.8rem; margin:0 0 20px 0; color:#0f172a !important;">The Gift Shop</h1>
            <div style="background-color: #f8fafc; border-left: 4px solid #0f172a; padding: 20px; margin: 0 0 30px 0;">
                <p style="font-size:15px; color:#334155 !important; font-style:italic; font-weight:500; margin:0 0 10px 0;">
                    "It started with a disaster: Gaurab brought a heavily scented, artisanal lavender candle to his cousin's birthday. His cousin is violently allergic to lavender."
                </p>
                <p style="font-size:15px; color:#334155 !important; font-style:italic; font-weight:500; margin:0 0 10px 0;">
                    "Meanwhile, Vidushi spent three exhaustive hours researching the 'ultimate kitchen gadget' only to buy a spatula that melted on day one. We realized gifting is broken—we overthink, overspend, and still miss the mark."
                </p>
                <p style="font-size:15px; color:#0f172a !important; font-style:italic; font-weight:800; margin:0;">
                    "That's why we built this AI engine—powered by the state-of-the-art Llama 3.1 model—to bring actual thought back into the art of giving."
                </p>
            </div>
            <p style="font-size:17px; color:#334155 !important; font-weight:500; margin:0 0 30px 0;">AI-powered gift recommendations for every occasion, milestone, and holiday.</p>
            <div style="display:flex; gap:20px;">
                <a onclick="document.getElementById('generator').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; background:#0f172a; color:#ffffff !important; padding:14px 32px; font-size:14px; font-weight:bold; text-transform:uppercase; border-radius:4px; text-decoration:none;">Start Gifting →</a>
                <a onclick="document.getElementById('blog').scrollIntoView({behavior: 'smooth'});" style="cursor:pointer; border:2px solid #0f172a; color:#0f172a !important; padding:14px 32px; font-size:14px; font-weight:bold; text-transform:uppercase; border-radius:4px; text-decoration:none;">Read the Blog</a>
            </div>
        </div>
        <div style="flex: 1 1 400px; position: relative;">
            <div style="position: relative; height: 450px; width: 100%;">
                <img src="https://images.unsplash.com/photo-1549465220-1a8b9238cd48?w=600&q=80" style="position: absolute; top: 0; left: 0; width: 70%; height: 300px; object-fit: cover; border-radius: 8px; box-shadow: 0 10px 30px rgba(0,0,0,0.15); animation: float 6s ease-in-out infinite;">
                <img src="https://images.unsplash.com/photo-1513885535751-8b9238bd345a?w=600&q=80" style="position: absolute; bottom: 0; right: 0; width: 65%; height: 250px; object-fit: cover; border-radius: 8px; box-shadow: 0 10px 30px rgba(0,0,0,0.15); border: 4px solid #fff; animation: float-delay 7s ease-in-out infinite;">
            </div>
        </div>
    </div>
    """)

    # ── GENERATOR SECTION ──
    gr.HTML("""<div id="generator" style="border-top:2px solid #e2e8f0; padding-top:60px; margin-bottom:40px;">
      <div style="font-size:12px; color:#10b981 !important; text-transform:uppercase; font-weight:800; letter-spacing:2px; margin-bottom:8px;">AI Generator</div>
      <h2 style="font-size:2.5rem; margin:0; color:#0f172a !important;">Find the Perfect Gift</h2>
    </div>""")

    with gr.Row():
        with gr.Column(scale=11):
            with gr.Row():
                q0 = gr.Dropdown(label="🎉 Occasion", choices=OCCASION_NAMES, value="🎂 Birthdays", interactive=True)
                q1 = gr.Dropdown(label="💰 Budget", choices=["Under $25","$25–$100","$100–$500","$500+"], value="$25–$100", interactive=True)
            with gr.Row():
                q2 = gr.Dropdown(label="👤 Recipient Persona", choices=["High-Tech","Outdoorsy","Homebody","Creative","Minimalist","Foodie"], value="Homebody", interactive=True)
                q3 = gr.Dropdown(label="🎯 Gift Priority", choices=["Practicality","Sentimental","Humor","Wow Factor","Experiences"], value="Practicality", interactive=True)
            user_context = gr.Textbox(label="Tell us about them", placeholder="Ex: She loves reading, works from home...", lines=4, interactive=True)
            submit_btn = gr.Button("✨ Generate Gift Ideas", variant="primary")

        with gr.Column(scale=9):
            hero_img_html = gr.HTML(update_hero_image("🎂 Birthdays"))

    output_html = gr.HTML("<p style='color:#64748b !important; font-weight:500; font-style:italic; padding:20px 0;'>Generated AI results will appear here...</p>")

    q0.change(fn=update_hero_image, inputs=q0, outputs=hero_img_html)
    submit_btn.click(fn=ui_logic_orchestrator, inputs=[q0,q1,q2,q3,user_context], outputs=output_html)

    # ── GIFTING PSYCHOLOGY & FUN FACTS (EXPANDED) ──
    gr.HTML("""
    <div style="margin-top:40px; margin-bottom:20px; padding-top: 40px; border-top: 1px dashed #cbd5e1;">
        <h3 style="font-size:22px; color:#0f172a !important; margin:0 0 20px 0; font-weight:900;">🧠 The Psychology of Gifting</h3>
        <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 20px;">
            <div style="background-color: #ffffff; padding: 20px; border-radius: 8px; border-left: 5px solid #10b981; box-shadow: 0 4px 15px rgba(0,0,0,0.03);">
                <h4 style="font-size:15px; color:#0f172a !important; margin:0 0 8px 0; font-weight:800;">1. The "Wow" Factor Myth</h4>
                <p style="font-size:13px; color:#334155 !important; line-height:1.6; margin:0;">Givers often optimize for the moment of opening, while receivers prefer long-term utility. A flashy gadget gets a gasp, but a high-quality coffee subscription gets gratitude every morning.</p>
            </div>
            <div style="background-color: #ffffff; padding: 20px; border-radius: 8px; border-left: 5px solid #3b82f6; box-shadow: 0 4px 15px rgba(0,0,0,0.03);">
                <h4 style="font-size:15px; color:#0f172a !important; margin:0 0 8px 0; font-weight:800;">2. The Price Tag Fallacy</h4>
                <p style="font-size:13px; color:#334155 !important; line-height:1.6; margin:0;">Research shows recipients don't correlate price with thoughtfulness. A $20 hyper-specific gift tailored to an obscure hobby always beats a generic $200 watch.</p>
            </div>
            <div style="background-color: #ffffff; padding: 20px; border-radius: 8px; border-left: 5px solid #f43f5e; box-shadow: 0 4px 15px rgba(0,0,0,0.03);">
                <h4 style="font-size:15px; color:#0f172a !important; margin:0 0 8px 0; font-weight:800;">3. The Experiential Advantage</h4>
                <p style="font-size:13px; color:#334155 !important; line-height:1.6; margin:0;">Psychologists found that experiential gifts—like a cooking class or concert tickets—create stronger social bonds and longer-lasting happiness than material objects.</p>
            </div>
            <div style="background-color: #ffffff; padding: 20px; border-radius: 8px; border-left: 5px solid #f59e0b; box-shadow: 0 4px 15px rgba(0,0,0,0.03);">
                <h4 style="font-size:15px; color:#0f172a !important; margin:0 0 8px 0; font-weight:800;">4. The "Just Because" Boost</h4>
                <p style="font-size:13px; color:#334155 !important; line-height:1.6; margin:0;">Receiving an unexpected gift on a random Tuesday releases more dopamine than holiday gifting, where expectations naturally modulate the brain's surprise response.</p>
            </div>
        </div>
    </div>
    """)

    # ── CART SECTION ──
    gr.HTML("""<div id="cart" style="border-top:2px solid #e2e8f0; padding-top:60px; margin-top:60px; margin-bottom:40px;">
      <div style="font-size:12px; color:#10b981 !important; text-transform:uppercase; font-weight:800; letter-spacing:2px; margin-bottom:8px;">🛒 Gift Cart</div>
      <h2 style="font-size:2.5rem; margin:0 0 8px; color:#0f172a !important;">Save Your Ideas</h2>
    </div>""")

    with gr.Row():
        with gr.Column(scale=3):
            c_occ = gr.Dropdown(label="Occasion", choices=OCCASION_NAMES, value="🎂 Birthdays", interactive=True)
            c_for = gr.Textbox(label="For Whom?", placeholder="Ex: Mom", interactive=True)
            c_idea = gr.Textbox(label="Gift Idea (Paste from AI)", placeholder="Ex: Ember Smart Mug", interactive=True)
            c_note = gr.Textbox(label="Notes", placeholder="Why this is good...", interactive=True)
            add_btn = gr.Button("➕ Add to Cart", variant="primary")
            cart_msg = gr.HTML("")

            gr.Markdown("<br><h3 style='color:#0f172a !important;'>Manage Cart</h3>")
            c_remove = gr.Textbox(label="Remove Item by ID", placeholder="Enter ID to remove", interactive=True)
            with gr.Row():
                rm_btn = gr.Button("🗑 Remove Item")
                clr_btn = gr.Button("⚠️ Clear All")

        with gr.Column(scale=9):
            cart_display = gr.HTML(render_cart())

    add_btn.click(fn=add_to_cart, inputs=[c_occ, c_for, c_idea, c_note], outputs=[cart_display, cart_msg])
    rm_btn.click(fn=remove_from_cart, inputs=c_remove, outputs=[cart_display, cart_msg])
    clr_btn.click(fn=clear_all_cart, inputs=[], outputs=[cart_display, cart_msg])

    # ── BLOG SECTION ──
    gr.HTML("""<div id="blog" style="border-top:2px solid #e2e8f0; padding-top:60px; margin-top:60px; margin-bottom:40px;">
      <div style="font-size:12px; color:#10b981 !important; text-transform:uppercase; font-weight:800; letter-spacing:2px; margin-bottom:8px;">The Blog</div>
      <h2 style="font-size:2.5rem; margin:0 0 24px; color:#0f172a !important;">Gifting Stories & Guides</h2>
    </div>""")

    with gr.Accordion("🌍 Read: The Global Gift-Giving Calendar (Holidays Around the World)", open=False):
        gr.HTML("""<img src="https://images.unsplash.com/photo-1528605248644-14dd04022da1?w=1200&q=80" style="width:100%; height:350px; object-fit:cover; border-radius:4px; margin-bottom:20px;">""")
        gr.Markdown("""
        **From Valentine's to Diwali to Lunar New Year — gift-giving is a universal language.**

        The best way to understand gifting is to understand the cultural context behind it. In **Lunar New Year**, the gift of the *Hongbao* (Red Envelope) isn't just about the money; the red paper itself symbolizes luck and wards off evil spirits. During **Diwali**, giving sweets (Mithai) and dry fruits represents a wish for prosperity and sweetness in the recipient's life.

        Understanding *why* a holiday is celebrated elevates your gift from an obligation to a deeply meaningful gesture.
        """)

    with gr.Accordion("🎓 Read: Milestone Gifts: When Stakes Are High", open=False):
        gr.HTML("""<img src="https://images.unsplash.com/photo-1511795409834-ef04bbd61622?w=1200&q=80" style="width:100%; height:350px; object-fit:cover; border-radius:4px; margin-bottom:20px;">""")
        gr.Markdown("""
        **Weddings, graduations, new babies — these moments deserve more than a gift card.**

        When a life milestone occurs, the recipient is standing on the threshold of a new chapter. The greatest milestone gifts act as tools for that next chapter.
        * **Graduations:** Buy them something that upgrades their professional life (a beautiful portfolio, a high-quality coffee machine for early mornings).
        * **Baby Showers:** Shift focus from the baby to the *parents*. Meal delivery services and high-quality insulated coffee mugs (because parents never get to finish hot coffee) are life-savers.
        """)

    with gr.Accordion("🧠 Read: Why We Get Gift-Giving Wrong (The Science of Gifting)", open=False):
        gr.HTML("""<img src="https://images.unsplash.com/photo-1549465220-1a8b9238cd48?w=1200&q=80" style="width:100%; height:350px; object-fit:cover; border-radius:4px; margin-bottom:20px;">""")
        gr.Markdown("""
        **Research shows givers consistently over-value expensive gifts. The science of generosity says presence and personalization win every time.**

        According to behavioral economists, there is a fundamental mismatch between givers and receivers. Givers want the "Wow!" factor at the moment the gift is opened. Receivers want *utility and long-term value*.

        **The Takeaway:** Stop worrying about the dramatic reveal. A highly practical gift that solves a minor annoyance in their daily life will be cherished long after the novelty of a flashy gift wears off.
        """)

    # ── CONTACT US SECTION (NEW) ──
    gr.HTML("""
    <div id="contact" style="border-top:2px solid #e2e8f0; padding-top:60px; margin-top:80px; margin-bottom:40px;">
      <div style="font-size:12px; color:#10b981 !important; text-transform:uppercase; font-weight:800; letter-spacing:2px; margin-bottom:8px;">Contact Us</div>
      <h2 style="font-size:2.5rem; margin:0 0 24px; color:#0f172a !important;">Need a Human Touch?</h2>
      <p style="font-size:16px; color:#334155 !important; margin-bottom: 40px; max-width:700px;">Sometimes an AI isn't enough. If you need bespoke curation, corporate gifting advice, or professional wrapping services, our network of experts is here to help.</p>

      <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 30px;">
          <div style="background:#ffffff; border:1px solid #cbd5e1; border-radius:8px; padding:30px; box-shadow:0 4px 15px rgba(0,0,0,0.03);">
              <h3 style="font-size:18px; color:#0f172a !important; margin:0 0 10px 0; font-weight:900;">🎁 Professional Gift Consultant</h3>
              <p style="font-size:14px; color:#475569 !important; margin-bottom:20px; line-height:1.6;">Stuck on a high-stakes gift like a 50th anniversary or a major corporate client? Let our senior consultant build a bespoke portfolio of options for you.</p>
              <p style="font-size:14px; font-weight:bold; color:#0f172a !important; margin:0;">✉️ Email: <a href="mailto:consulting@thegiftshop.ai" style="color:#0284c7; text-decoration:none;">consulting@thegiftshop.ai</a></p>
          </div>

          <div style="background:#ffffff; border:1px solid #cbd5e1; border-radius:8px; padding:30px; box-shadow:0 4px 15px rgba(0,0,0,0.03);">
              <h3 style="font-size:18px; color:#0f172a !important; margin:0 0 10px 0; font-weight:900;">🎀 Master Gift Wrapper</h3>
              <p style="font-size:14px; color:#475569 !important; margin-bottom:20px; line-height:1.6;">Presentation is 50% of the gift. Our master wrappers use imported Japanese paper, silk ribbons, and wax seals to make your gift unforgettable.</p>
              <p style="font-size:14px; font-weight:bold; color:#0f172a !important; margin:0;">✉️ Email: <a href="mailto:wrapping@thegiftshop.ai" style="color:#0284c7; text-decoration:none;">wrapping@thegiftshop.ai</a></p>
          </div>

          <div style="background:#0f172a; border-radius:8px; padding:30px; box-shadow:0 4px 15px rgba(0,0,0,0.1);">
              <h3 style="font-size:18px; color:#ffffff !important; margin:0 0 10px 0; font-weight:900;">💬 General Inquiries</h3>
              <p style="font-size:14px; color:#cbd5e1 !important; margin-bottom:20px; line-height:1.6;">Have questions about the AI generator? Found a bug? Just want to say hi to Gaurab and Vidushi? We'd love to hear from you.</p>
              <p style="font-size:14px; font-weight:bold; color:#ffffff !important; margin:0;">✉️ Email: <a href="mailto:hello@thegiftshop.ai" style="color:#38bdf8; text-decoration:none;">hello@thegiftshop.ai</a></p>
          </div>
      </div>
    </div>
    """)

    # ── FOOTER ──
    gr.HTML("""
    <div style="border-top:2px solid #0f172a; padding:40px 0; text-align:center; margin-top:80px;">
      <div style="font-size:24px; font-weight:900; color:#0f172a !important; margin-bottom:10px;">🎁 The Gift Shop</div>
      <div style="font-size:15px; color:#334155 !important; font-weight:600;">Created by <strong>Gaurab & Vidushi</strong> | Final AI Project — <strong>Prof. Armen Pischsdotchian</strong></div>
    </div>
    """)

if __name__ == "__main__":
    demo.launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
✅ All packages ready! Starting the app...


/tmp/ipykernel_8237/1168305463.py:313: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=luxury_css, js=js_func) as demo:
/tmp/ipykernel_8237/1168305463.py:313: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(css=luxury_css, js=js_func) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c6fc724958512339e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
